# 02 — Pooled XGBoost RUL Training on FD001–FD004

This is the main training notebook. It trains **one XGBoost Regressor** on the complete run-to-failure training trajectories from all four NASA C-MAPSS subsets. It does **not** train on official test labels.

Robustness choices:
- retain all 21 sensors and all 3 operating settings;
- create trailing 5-cycle mean, standard deviation, and one-cycle-change features without looking into the future;
- one-hot encode the subset identity (FD001–FD004) so the model can distinguish simulated regimes;
- hold out entire engines within each subset for pseudo-truncated validation;
- use GroupKFold by `engine_key` during parameter selection;
- refit the selected XGBoost configuration on every available training engine before saving the dashboard artifact.

Training RUL is capped at 125 cycles, a common C-MAPSS convention. Therefore, report both capped-target and uncapped official-test metrics in Notebook 03.

In [ ]:
from pathlib import Path
import zipfile
import shutil
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
REQUIRED_FILES = [f"{kind}_{subset}.txt" for subset in SUBSETS for kind in ("train", "test", "RUL")]
DATASET_URLS = [
    "https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
    "https://zenodo.org/records/15346912/files/CMAPSSData.zip?download=1",
    "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
]

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def find_dataset_file(filename):
    """Look in data/raw first, then an extracted CMAPSSData folder, then project root."""
    candidates = [RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def ensure_full_dataset(download_if_missing=True):
    missing = [name for name in REQUIRED_FILES if find_dataset_file(name) is None]
    if not missing:
        print("All 12 C-MAPSS files found across data/raw / CMAPSSData / project root.")
        return
    if not download_if_missing:
        raise FileNotFoundError(
            "Missing C-MAPSS files: " + ", ".join(missing) +
            ". Copy all train_FD00x.txt, test_FD00x.txt and RUL_FD00x.txt files into data/raw/ or extract CMAPSSData.zip into the project root."
        )
    archive_path = RAW_DIR / "CMAPSSData.zip"
    last_error = None
    downloaded = False
    for url in DATASET_URLS:
        try:
            print(f"Attempting to download the full C-MAPSS archive from {url.split('/')[2]} ...")
            temporary_path = archive_path.with_suffix(".zip.part")
            with requests.get(url, stream=True, timeout=(20, 180)) as response:
                response.raise_for_status()
                with open(temporary_path, "wb") as out:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            out.write(chunk)
            if not zipfile.is_zipfile(temporary_path):
                temporary_path.unlink(missing_ok=True)
                raise ValueError("The server response was not a valid ZIP archive.")
            temporary_path.replace(archive_path)
            downloaded = True
            break
        except Exception as exc:
            last_error = exc
            print(f"Download attempt failed: {type(exc).__name__}: {exc}")
    if not downloaded:
        raise RuntimeError(
            "Automatic download did not succeed. The full archive is available from the NASA PCoE repository: "
            "https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/ . "
            "Download '6. Turbofan Engine Degradation Simulation Data Set', extract it, and copy all 12 required .txt files to data/raw/. "
            f"Last error: {last_error}"
        )
    with zipfile.ZipFile(archive_path) as archive:
        members = {Path(member).name: member for member in archive.namelist() if not member.endswith("/")}
        not_in_archive = [name for name in REQUIRED_FILES if name not in members]
        if not_in_archive:
            raise FileNotFoundError(f"Archive is missing expected files: {not_in_archive}")
        for filename in REQUIRED_FILES:
            with archive.open(members[filename]) as src, open(RAW_DIR / filename, "wb") as dst:
                shutil.copyfileobj(src, dst)
    print(f"Extracted all C-MAPSS files to {RAW_DIR}")


def load_cmapss_file(filename):
    path = find_dataset_file(filename)
    if path is None:
        raise FileNotFoundError(f"{filename} was not found. Run ensure_full_dataset() or follow README.md.")
    frame = pd.read_csv(path, sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected column count in {path}: {frame.shape[1]}")
    return frame


def load_all_subsets(include_test=True):
    train_frames, test_frames, rul_frames = [], [], []
    for subset in SUBSETS:
        train = load_cmapss_file(f"train_{subset}.txt")
        train["subset"] = subset
        train["engine_key"] = subset + "_" + train["unit_nr"].astype(int).astype(str).str.zfill(3)
        train_frames.append(train)
        if include_test:
            test = load_cmapss_file(f"test_{subset}.txt")
            test["subset"] = subset
            test["engine_key"] = subset + "_" + test["unit_nr"].astype(int).astype(str).str.zfill(3)
            test_frames.append(test)
            rul_path = find_dataset_file(f"RUL_{subset}.txt")
            if rul_path is None:
                raise FileNotFoundError(f"RUL_{subset}.txt is missing.")
            rul_vector = pd.read_csv(rul_path, header=None).iloc[:, 0].to_numpy(dtype=float)
            test_ids = np.sort(test["unit_nr"].unique().astype(int))
            if len(test_ids) != len(rul_vector):
                raise ValueError(f"{subset}: {len(test_ids)} test engines but {len(rul_vector)} RUL labels.")
            rul_frames.append(pd.DataFrame({
                "subset": subset,
                "unit_nr": test_ids,
                "engine_key": [f"{subset}_{u:03d}" for u in test_ids],
                "true_rul_cycles": rul_vector,
            }))
    train_all = pd.concat(train_frames, ignore_index=True)
    if not include_test:
        return train_all
    return train_all, pd.concat(test_frames, ignore_index=True), pd.concat(rul_frames, ignore_index=True)


import json
import joblib
from sklearn.model_selection import train_test_split, GroupKFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

ensure_full_dataset(download_if_missing=False)
train_raw = load_all_subsets(include_test=False)
print("Training rows across all subsets:", f"{len(train_raw):,}")
print("Training engines:", f"{train_raw['engine_key'].nunique():,}")
display(train_raw.groupby("subset").agg(rows=("time_cycles","size"), engines=("engine_key","nunique")))

## 1. Target and causal feature engineering

For each training engine, `RUL_true = final_cycle − current_cycle`. The fitting target is `min(RUL_true, 125)`. `engine_key` is used for grouping only and never enters the predictor matrix.

In [ ]:
ROLLING_WINDOW = 5
RUL_CAP = 125
RANDOM_STATE = 42
SENSOR_COLS = [f"s_{i}" for i in range(1, 22)]
SETTING_COLS = ["setting_1", "setting_2", "setting_3"]
SUBSET_FEATURES = [f"subset_{subset}" for subset in SUBSETS]


def engineer_features(frame, window=ROLLING_WINDOW):
    """Add current, trailing-window and delta features per (subset, engine)."""
    ordered = frame.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
    required = ["subset", "unit_nr", "engine_key", "time_cycles"] + SETTING_COLS + SENSOR_COLS
    absent = [column for column in required if column not in ordered.columns]
    if absent:
        raise ValueError(f"Missing columns for feature engineering: {absent}")
    features = ordered[required].copy()
    groups = ordered.groupby(["subset", "unit_nr"], sort=False)
    for sensor in SENSOR_COLS:
        signal = groups[sensor]
        features[f"{sensor}_roll_mean_{window}"] = signal.transform(
            lambda series: series.rolling(window=window, min_periods=1).mean()
        )
        features[f"{sensor}_roll_std_{window}"] = signal.transform(
            lambda series: series.rolling(window=window, min_periods=2).std(ddof=0)
        ).fillna(0.0)
        features[f"{sensor}_delta_1"] = signal.diff().fillna(0.0)
    for subset in SUBSETS:
        features[f"subset_{subset}"] = (ordered["subset"] == subset).astype(np.int8)
    return features


train_ordered = train_raw.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
train_ordered["RUL_true"] = (
    train_ordered.groupby(["subset", "unit_nr"])["time_cycles"].transform("max")
    - train_ordered["time_cycles"]
).astype(float)
train_ordered["RUL_target"] = train_ordered["RUL_true"].clip(upper=RUL_CAP)
train_features = engineer_features(train_ordered)
train_features["RUL_true"] = train_ordered["RUL_true"].to_numpy()
train_features["RUL_target"] = train_ordered["RUL_target"].to_numpy()
DROP_FROM_X = {"subset", "unit_nr", "engine_key", "RUL_true", "RUL_target"}
FEATURE_COLUMNS = [column for column in train_features.columns if column not in DROP_FROM_X]

assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS))
assert not train_features[FEATURE_COLUMNS].isna().any().any(), "Unexpected missing predictor values"
assert "unit_nr" not in FEATURE_COLUMNS and "engine_key" not in FEATURE_COLUMNS
print("Pooled feature matrix:", train_features[FEATURE_COLUMNS].shape)
print("Total predictors:", len(FEATURE_COLUMNS))
print("Sensors retained:", len(SENSOR_COLS), "| Settings retained:", len(SETTING_COLS))
print("RUL training cap:", RUL_CAP, "cycles")
display(train_features[["subset", "engine_key", "time_cycles", "RUL_true", "RUL_target"] + SUBSET_FEATURES].head())

## 2. Engine-aware development / validation split

We hold out 20% of engines separately inside each subset. This guarantees every subset appears in both development and validation, while all cycles from an individual engine stay on one side of the split.

In [ ]:
DEVELOPMENT_KEYS, VALIDATION_KEYS = [], []
for subset in SUBSETS:
    engine_keys = np.sort(train_features.loc[train_features["subset"] == subset, "engine_key"].unique())
    dev_keys, val_keys = train_test_split(
        engine_keys, test_size=0.20, random_state=RANDOM_STATE
    )
    DEVELOPMENT_KEYS.extend(dev_keys.tolist())
    VALIDATION_KEYS.extend(val_keys.tolist())

is_dev = train_features["engine_key"].isin(DEVELOPMENT_KEYS)
is_val = train_features["engine_key"].isin(VALIDATION_KEYS)
X_dev = train_features.loc[is_dev, FEATURE_COLUMNS]
y_dev = train_features.loc[is_dev, "RUL_target"].astype(float)
g_dev = train_features.loc[is_dev, "engine_key"].astype(str)
val_rows = train_features.loc[is_val].copy()

split_summary = []
for subset in SUBSETS:
    split_summary.append({
        "subset": subset,
        "development_engines": train_features.loc[is_dev & (train_features["subset"] == subset), "engine_key"].nunique(),
        "validation_engines": train_features.loc[is_val & (train_features["subset"] == subset), "engine_key"].nunique(),
    })
display(pd.DataFrame(split_summary))
print(f"Development rows: {len(X_dev):,} | held-out validation rows: {len(val_rows):,}")

## 3. Compact, group-aware XGBoost hyperparameter search

The search uses three-fold `GroupKFold` on development engines, optimising MAE on the capped target. This is deliberately a small search so the project remains feasible on a student laptop. XGBoost remains the only predictive algorithm family.

In [ ]:
param_distributions = {
    "n_estimators": [250, 350, 450, 550],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05, 0.08],
    "subsample": [0.80, 0.90, 1.00],
    "colsample_bytree": [0.80, 0.90, 1.00],
    "min_child_weight": [1, 3, 5],
    "reg_lambda": [1.0, 3.0, 7.0],
}
base_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=4,
)
search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="neg_mean_absolute_error",
    cv=GroupKFold(n_splits=3),
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=1,
    refit=True,
    return_train_score=True,
    error_score="raise",
)
search.fit(X_dev, y_dev, groups=g_dev)
print("Best grouped-CV MAE on capped target:", round(-search.best_score_, 3), "cycles")
print("Best XGBoost parameters:")
print(json.dumps(search.best_params_, indent=2))
cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cv_results.to_csv(REPORTS_DIR / "all_subsets_xgboost_group_cv_results.csv", index=False)
display(cv_results[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(6))

## 4. Pseudo-truncated validation across held-out engines

For each held-out validation engine, choose one point before its known end-of-life and predict RUL from the sensor history observed by that point. This avoids the invalid evaluation where every complete training engine has RUL = 0 at its last cycle. Scores here are validation results only; Notebook 03 evaluates the untouched official test sets.

In [ ]:
def nasa_asymmetric_score(y_true, y_pred):
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    penalties = np.where(
        error < 0,
        np.exp(np.clip(-error / 13.0, 0, 50)) - 1,
        np.exp(np.clip(error / 10.0, 0, 50)) - 1,
    )
    return float(np.sum(penalties))


def metric_pack(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(0.0, np.asarray(y_pred, dtype=float))
    return {
        "MAE_cycles": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_cycles": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "NASA_asymmetric_score_lower_is_better": nasa_asymmetric_score(y_true, y_pred),
        "early_predictions_pct": float(np.mean(y_pred < y_true) * 100),
        "late_predictions_pct": float(np.mean(y_pred > y_true) * 100),
        "n_validation_engines": int(len(y_true)),
    }

rng = np.random.default_rng(RANDOM_STATE)
pseudo_indices = []
for engine_key, engine_rows in val_rows.groupby("engine_key", sort=True):
    final_cycle = int(engine_rows["time_cycles"].max())
    max_simulated_rul = min(RUL_CAP, final_cycle - (ROLLING_WINDOW + 1))
    if max_simulated_rul < 1:
        continue
    min_simulated_rul = min(5, max_simulated_rul)
    simulated_rul = int(rng.integers(min_simulated_rul, max_simulated_rul + 1))
    cutoff_cycle = final_cycle - simulated_rul
    chosen = engine_rows.loc[engine_rows["time_cycles"] == cutoff_cycle]
    if chosen.empty:
        chosen = engine_rows.loc[engine_rows["time_cycles"] <= cutoff_cycle].tail(1)
    if not chosen.empty:
        pseudo_indices.append(chosen.index[-1])

val_pseudo = val_rows.loc[pseudo_indices].sort_values(["subset", "unit_nr"]).reset_index(drop=True)
y_val_true = val_pseudo["RUL_true"].to_numpy(dtype=float)
pred_val = np.maximum(0.0, search.best_estimator_.predict(val_pseudo[FEATURE_COLUMNS]))
y_val_capped = np.minimum(y_val_true, RUL_CAP)
pred_val_capped = np.clip(pred_val, 0.0, RUL_CAP)
validation_summary = {
    "uncapped_rul_validation": metric_pack(y_val_true, pred_val),
    "capped_125_target_validation": metric_pack(y_val_capped, pred_val_capped),
    "validation_note": "Held-out training engines at simulated truncation points; not the official test set.",
}
print(json.dumps(validation_summary, indent=2))
validation_predictions = val_pseudo[["subset", "unit_nr", "engine_key", "time_cycles"]].rename(columns={"time_cycles":"validation_cutoff_cycle"}).copy()
validation_predictions["true_rul_cycles"] = y_val_true
validation_predictions["predicted_rul_cycles"] = pred_val
validation_predictions["error_pred_minus_true"] = pred_val - y_val_true
validation_predictions.to_csv(REPORTS_DIR / "all_subsets_validation_predictions.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_val_true, pred_val, alpha=0.75, c=pd.Categorical(val_pseudo["subset"], categories=SUBSETS).codes)
limit = float(max(y_val_true.max(), pred_val.max(), 1))
axes[0].plot([0, limit], [0, limit], "--", color="gray")
axes[0].set(title="Held-out pseudo-test: predicted vs true RUL", xlabel="True RUL (cycles)", ylabel="Predicted RUL (cycles)")
axes[1].hist(pred_val - y_val_true, bins=22, edgecolor="white")
axes[1].axvline(0, linestyle="--", color="gray")
axes[1].set(title="Validation errors", xlabel="Predicted − true RUL (cycles)", ylabel="Engines")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "all_subsets_validation.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Refit on the complete training dataset and save the new artifact

After hyperparameter selection and validation, fit the final XGBoost model on **all 709 available training engines across FD001–FD004**. This is the final training procedure: all labelled run-to-failure training rows are used, while official test trajectories and their RUL labels remain excluded from fitting.

In [ ]:
final_model = XGBRegressor(
    **search.best_params_,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=4,
)
X_all = train_features[FEATURE_COLUMNS]
y_all = train_features["RUL_target"].astype(float)
final_model.fit(X_all, y_all)

artifact = {
    "model": final_model,
    "sensor_cols": SENSOR_COLS,
    "setting_cols": SETTING_COLS,
    "feature_columns": FEATURE_COLUMNS,
    "rolling_window": ROLLING_WINDOW,
    "rul_cap": RUL_CAP,
    "best_params": search.best_params_,
    "dataset": "NASA PCoE C-MAPSS FD001-FD004 pooled model",
    "subsets": SUBSETS,
    "target_unit": "operating cycles",
    "random_state": RANDOM_STATE,
    "n_training_rows": int(len(train_raw)),
    "n_training_engines": int(train_raw["engine_key"].nunique()),
    "n_model_features": int(len(FEATURE_COLUMNS)),
    "target_policy": "RUL_true = final_cycle - current_cycle; fit target clipped at 125 cycles",
}
model_path = MODELS_DIR / "xgb_rul_model_all_subsets.joblib"
joblib.dump(artifact, model_path)

training_summary = {
    "dataset": artifact["dataset"],
    "algorithm": "XGBoost Regressor only",
    "training_rows": int(len(train_raw)),
    "training_engines": int(train_raw["engine_key"].nunique()),
    "training_rows_by_subset": {s: int((train_raw["subset"] == s).sum()) for s in SUBSETS},
    "training_engines_by_subset": {s: int(train_raw.loc[train_raw["subset"] == s, "engine_key"].nunique()) for s in SUBSETS},
    "n_model_features": int(len(FEATURE_COLUMNS)),
    "sensor_channels_retained": SENSOR_COLS,
    "operating_settings_retained": SETTING_COLS,
    "subset_one_hot_features": SUBSET_FEATURES,
    "rul_training_cap_cycles": RUL_CAP,
    "grouped_cv_best_mae_capped_target_cycles": float(-search.best_score_),
    "pseudo_truncated_validation": validation_summary,
    "best_params": search.best_params_,
    "official_test_data_used_for_fit": False,
    "model_artifact": str(model_path),
}
(REPORTS_DIR / "all_subsets_training_summary.json").write_text(json.dumps(training_summary, indent=2))
print("Saved full-dataset model:", model_path)
print("Training engines:", artifact["n_training_engines"], "| rows:", f"{artifact['n_training_rows']:,}")
print("Next: run notebooks/03_test_evaluation_and_explainability.ipynb")

## Interview talking points

- **Why all four subsets?** They introduce multiple simulated operating regimes and fault modes; the pooled model tests whether one model can use those measurements and a subset indicator.
- **Why one-hot subset indicators?** FD001–FD004 are categories, not a numeric ranking.
- **Why GroupKFold?** A given engine's cycles are dependent; every engine must belong to exactly one CV fold.
- **Why the 125-cycle target cap?** It focuses fitting on degradation nearer to failure. It also limits predictions above 125, so uncapped metrics must be reported as a limitation, not hidden.
- **What does “trained on the complete dataset” mean?** The final fit uses every run-to-failure training trajectory across all four subsets. The official test files and labels remain unseen during fit and model selection.